# Dynamo: parameter optim.
## Setup

In [2]:
import dynamo as dyn
import numpy as np
import anndata as ad

/mnt/mib-nas01/ohkawalab/maehara/work/ddhodge2025/envs/dynamo/lib/python3.11/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
  warnings.warn(msg, errors.NumbaWarning)
/mnt/mib-nas01/ohkawalab/maehara/work/ddhodge2025/envs/dynamo/lib/python3.11/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
  warnings.warn(msg, errors.NumbaWarning)
/mnt/mib-nas01/ohkawalab/maehara/work/ddhodge2025/envs/dynamo/lib/python3.11/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates 

## Generate test data

In [4]:
datadir = "../benchmark/dynamo/datasets"
for N in [100, 500, 1000, 5000, 10000, 50000, 100000]:
    file = datadir + f"/twogenes_N{N}.h5ad"
    print(file)
    adata = dyn.sim.Simulator(motif="twogenes", cell_num=N)
    adata.write(datadir + f"/twogenes_N{N}.h5ad")

../benchmark/dynamo/datasets/twogenes_N100.h5ad
../benchmark/dynamo/datasets/twogenes_N500.h5ad
../benchmark/dynamo/datasets/twogenes_N1000.h5ad
../benchmark/dynamo/datasets/twogenes_N5000.h5ad
../benchmark/dynamo/datasets/twogenes_N10000.h5ad
../benchmark/dynamo/datasets/twogenes_N50000.h5ad
../benchmark/dynamo/datasets/twogenes_N100000.h5ad


## Load data

In [6]:
N = 5000
#N = 100000
adata = ad.read_h5ad(datadir+f"/twogenes_N{N}.h5ad")
#adata = ad.read_h5ad("../data/sim3D_lorenz.h5ad"); N = adata.n_obs
adata.obsm['X_pca'], adata.obsm['velocity_pca'] = adata.X, adata.layers['velocity']
adata.var['use_for_dynamics'] = True # i.e., use all
adata.uns['PCs'] = np.identity(adata.X.shape[1]) # bypass PCA i.e., original space
#adata.var['gamma'] = 1
adata

AnnData object with n_obs × n_vars = 5000 × 2
    var: 'use_for_dynamics'
    uns: 'PCs'
    obsm: 'X_pca', 'velocity_pca'
    layers: 'velocity'

## Run dynamo

In [ ]:
dyn.vf.VectorField(
    adata, basis='pca', velocity_key='velocity', pot_curl_div=False,
    M=100, MaxIter=5000, lstsq_method='scipy', div_cur_free_kernels=True,
)
dyn.vf.jacobian(adata, basis='pca', store_in_adata=True)
dyn.vf.curl(adata,basis='pca')
dyn.vf.divergence(adata,basis='pca')
#adata.write("../data/surfsim_shpere.h5ad")
adata

|-----> VectorField reconstruction begins...
|-----> Retrieve X and V based on basis: PCA. 
        Vector field will be learned in the PCA space.
|-----> Generating high dimensional grids and convert into a row matrix.
|-----> Learning vector field with method: sparsevfc.
|-----> [SparseVFC] begins...
|-----> Sampling control points based on data velocity magnitude...
|-----> method arg is None, choosing methods automatically...
|-----------> method kd_tree selected


Iterating each dimension in con_K_div_cur_free:: 100%|██████████| 2/2 [00:00<00:00, 45.81it/s]


In [135]:
adata.uns['jacobian_pca']['jacobian'][:,:,1]

array([[-0.85379002, -0.12117719],
       [-0.14745321, -0.8739637 ]])

## Analytical values

In [136]:
from dynamo.vectorfield.utils import compute_curl, compute_divergence
from dynamo.simulation.ODE import two_genes_motif_jacobian
#(x, t=None, a1=1, a2=1, b1=1, b2=1, k1=1, k2=1, S=0.5, n=4)

X = adata.X

def f_jac(X):
    if X.ndim == 1: X = X.reshape((1, -1))
    J = np.zeros((X.shape[1], X.shape[1], X.shape[0]))
    for ind, i in enumerate(X):
        J[:, :, ind] = two_genes_motif_jacobian(i[0], i[1])
    return J

analytical_curl = compute_curl(f_jac, X)
analytical_div = compute_divergence(f_jac, X, vectorize_size=1)
adata.obs['analytical_curl'] = analytical_curl.copy()
adata.obs['analytical_div'] = analytical_div.copy()

# set up the dictionary for the true jacobian
J_dict = adata.uns['jacobian_pca'].copy()

J = np.zeros_like(J_dict['jacobian'])
for ind, i in enumerate(adata.X):
    J[:, :, ind] = two_genes_motif_jacobian(i[0], i[1])

J_dict['jacobian'] = J
adata.uns['jacobian_true'] = J_dict

adata

Calculating divergence: 100%|██████████| 100000/100000 [00:01<00:00, 93046.32it/s]


AnnData object with n_obs × n_vars = 100000 × 2
    obs: 'control_point_pca', 'inlier_prob_pca', 'obs_vf_angle_pca', 'jacobian_det_pca', 'curl_pca', 'divergence_pca', 'analytical_curl', 'analytical_div'
    var: 'use_for_dynamics'
    uns: 'PCs', 'VecFld_pca', 'jacobian_pca', 'jacobian_true'
    obsm: 'X_pca', 'velocity_pca', 'velocity_pca_SparseVFC', 'X_pca_SparseVFC'
    layers: 'velocity'

In [137]:
def eval_mses(adata):
    mse = lambda x, y: np.mean((x - y)**2)
    J_a = f_jac(adata.X)
    curl_a = compute_curl(f_jac, adata.X)
    div_a = compute_divergence(f_jac, adata.X, vectorize_size=1)
    detJ_a = [np.linalg.det(J_a[:,:,i]) for i in range(adata.n_obs)]
    return {
        'mse_div': mse(adata.obs.divergence_pca, div_a),
        'mse_curl': mse(adata.obs.curl_pca, curl_a),
        'mse_detJ': mse(adata.obs.jacobian_det_pca, detJ_a)
    }

eval_mses(adata)

Calculating divergence: 100%|██████████| 100000/100000 [00:01<00:00, 80115.97it/s]


{'mse_div': 0.011472855682545982,
 'mse_curl': 0.011089529274947968,
 'mse_detJ': 0.011336088185106128}

## Save data

In [138]:
#adata.write("../data/dynamo_toggle_2025revise_scipy_M1000.h5ad")
#adata.write("../data/dynamo_toggle_2025revise_default.h5ad")